# Prepare a private household HASS dataset
Global settings come from the visible `training.env` next to this notebook. All generation, validation, grouping and Parquet writing run in Dagster. This notebook uploads one snapshot and displays metadata/examples. The bounded pilot remains unchanged; the larger cut requests up to 1,200 episodes across grounded individual light/switch tasks. Actual counts depend on the inventory. No canonical moderation approval is implied.
Before starting, deploy the pipeline image with `prepare_hass_household_cut` and `export_hass_household_cut`. Generate a full Assist snapshot in HA using the action below, download `/config/assist_snapshots/<capture-uuid>.json`, and upload it to Jupyter. Reuse your stable household UUID.

In [ ]:
from pathlib import Path
import sys, json, hashlib, re
WORKSPACE = Path.home() / 'eliza/training'
DATASET_LABEL = 'hass-household-01'
CAPTURE_FILE = ''
HOUSEHOLD_UUID = ''
SOURCE_BUCKET = 'synthetic'
DESTINATION_BUCKET = 'sft-datasets'
MAX_EPISODES = 1200
SEED = 42
if (WORKSPACE / '.client/nlp_training').is_dir():
    sys.path.insert(0, str(WORKSPACE / '.client'))
from nlp_training.operator import connect_workspace
from eliza_hass import HassPilotWorkflow
settings, storage, training, dagster = connect_workspace(WORKSPACE)
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', DATASET_LABEL):
    raise ValueError('Use an alphanumeric dataset label')
RUN = WORKSPACE / 'runs' / DATASET_LABEL
RUN.mkdir(parents=True, exist_ok=True)
progress_file = RUN / 'dataset-progress.json'
progress = json.loads(progress_file.read_text()) if progress_file.exists() else {}
def remember(**values):
    if any(key in progress and progress[key] != value for key, value in values.items()):
        raise ValueError('This dataset label already binds a different capture/configuration')
    progress.update(values)
    progress_file.write_text(json.dumps(progress, indent=2))
def launch(stage, job, op, config):
    import yaml
    old = progress.get(stage)
    if old:
        run = dagster.get_run(old)
        submitted = yaml.safe_load(run.run_config_yaml)['ops'][op]['config']
        if run.job_name != job or submitted != config:
            raise ValueError('Saved run configuration differs; use a new dataset label')
        return old
    run_id = dagster.launch_run(job, {'ops': {op: {'config': config}}})
    remember(**{stage: run_id})
    return run_id
def finish(run_id, op):
    dagster.watch_run(run_id)
    run = dagster.get_run(run_id)
    if run.status != 'SUCCESS':
        raise RuntimeError(f'Dagster run {run_id} is {run.status}')
    return dagster.output_metadata(run_id, step_key=op)
print('Dataset run records:', RUN)


## Capture and ingest
Run this in HA Developer Tools → Actions (YAML), then upload the saved JSON and set CAPTURE_FILE. No HA token belongs in this notebook.
```yaml
action: assist_snapshot.capture
data:
  mode: full
  api_ids: [assist]
  platform: llama_conversation
  inventory_scope: exposed
  allow_runtime_side_effects: true
  save: true
```
The next cell archives the exact file, launches credential redaction in Dagster and resolves the exact fixture. Run IDs persist in the dataset folder.

In [ ]:
pilot = HassPilotWorkflow(source_id=HOUSEHOLD_UUID, bucket=SOURCE_BUCKET, storage=storage, dagster=dagster)
pilot.check_jobs()
remember(household_uuid=HOUSEHOLD_UUID, source_bucket=SOURCE_BUCKET)
if progress.get('source_key'):
    pilot.use_archived_source(progress['source_key'])
if CAPTURE_FILE:
    pilot.archive_file(CAPTURE_FILE)
remember(source_key=pilot.source_key)
pilot.ingest_run_id = progress.get('ingest_run_id')
remember(ingest_run_id=pilot.launch_ingestion())
pilot.watch('ingest')
fixture = pilot.resolve_fixture()
print('Fixture:', fixture['fixture_key'])


## Generate, validate and preview
Dagster expands valid entities, actions, state/failure conditions, numeric brightness values, compound pairs and clarification cases. The requested count is a maximum, not a quota padded with duplicates. The split keeps each base-task group together, targets approximately 20% held-out episodes and includes calls and grounded replies in eval. It is a within-household benchmark. This version uses templated English wording and simulator outcomes, not live executions.

In [ ]:
config = dict(bucket=SOURCE_BUCKET, fixture_key=fixture['fixture_key'], seed=SEED, max_episodes=MAX_EPISODES)
prepared_id = launch('prepare_run_id', 'prepare_hass_household_cut', 'prepare_hass_household_cut_op', config)
prepared = finish(prepared_id, 'prepare_hass_household_cut_op')
payload = storage.read_bytes(bucket=SOURCE_BUCKET, key=prepared['preview_key'], max_bytes=32*1024*1024)
if hashlib.sha256(payload).hexdigest() != prepared['preview_sha256']:
    raise ValueError('Preview bytes changed')
preview = json.loads(payload)
(RUN / 'dataset-preview.json').write_bytes(payload)
print('Requested:', preview['requested_episodes'], '| Actual:', preview['actual_episodes'], '| Distinct groups:', preview['distinct_groups'])
print('Families:', preview['families'])
print('Splits:', preview['splits'])
print(*preview['notes'], sep='\n')


In [ ]:
for split, rows in preview['examples'].items():
    for row in rows:
        print('---', split, row['family'], row['example_id'])
        for message in row['messages'][1:]:
            print(message['role'], message.get('content'), message.get('tool_calls', ''))
        print('TARGET:', row['target_message'])
# Exact system prompt and advertised schemas are in preview['examples'][split][row_number].


## Freeze and export the previewed cut
Run this cell to export the exact immutable index. The worker replays the recipe and checks all source hashes, then publishes bounded Parquet shards followed by the manifest. Running again reuses identical objects. `release.json` is the receipt selected by DATASET_LABEL in the training notebook; all three model runs should use this same release.

In [ ]:
config = dict(bucket=SOURCE_BUCKET, destination_bucket=DESTINATION_BUCKET, dataset_id='hass_household_private',
              index_key=prepared['index_key'], index_sha256=prepared['index_sha256'], expected_cut_id=prepared['cut_id'])
export_id = launch('export_run_id', 'export_hass_household_cut', 'export_hass_household_cut_op', config)
release = finish(export_id, 'export_hass_household_cut_op')
(RUN / 'release.json').write_text(json.dumps(release, indent=2))
release
